# EDA Graph Debugging

Use this notebook to debug the LangGraph routing around the EDA step.

This notebook is set up for two kinds of checks:

1. static graph-shape inspection
2. dry-run node inspection for the EDA-stage agents using mocked LLM responses


In [1]:
from pprint import pprint

from multi_agent_ds.adapters.agent_frameworks.langgraph import (
    get_debug_graph,
    render_graph_ascii,
    render_graph_mermaid,
)
from multi_agent_ds.orchestration.graph import build_graph

compiled = build_graph().compile()
debug_graph = get_debug_graph(compiled, xray=True)

ascii_graph = render_graph_ascii(compiled, xray=True)
mermaid_graph = render_graph_mermaid(compiled, xray=True)

print(ascii_graph)


                                                                              +-----------+                                                            
                                                                              | __start__ |                                                            
                                                                              +-----------+                                                            
                                                                                     *                                                                 
                                                                                     *                                                                 
                                                                                     *                                                                 
                                                                                +-------

In [4]:
edges = [(edge.source, edge.target) for edge in debug_graph.edges]

raw_targets = sorted(target for source, target in edges if source == 'eda_raw')
processed_targets = sorted(target for source, target in edges if source == 'eda_processed')
prep_plan_inputs = sorted(source for source, target in edges if target == 'eda_prep_plan')
processed_approval_inputs = sorted(source for source, target in edges if target == 'eda_processed_approval')

print('Raw fan-out targets:')
pprint(raw_targets)
print('\nProcessed fan-out targets:')
pprint(processed_targets)
print('\nFan-in to eda_prep_plan:')
pprint(prep_plan_inputs)
print('\nFan-in to eda_processed_approval:')
pprint(processed_approval_inputs)

assert raw_targets == [
    'business_stakeholder_raw_review',
    'ml_modeler_raw_review',
    'ml_reviewer_raw_review',
]
assert processed_targets == [
    'business_stakeholder_processed_review',
    'ml_modeler_processed_review',
    'ml_reviewer_processed_review',
]
assert prep_plan_inputs == [
    'business_stakeholder_raw_review',
    'data_engineer_feedback',
    'eda_processed_approval',
    'ml_modeler_raw_review',
    'ml_reviewer_raw_review',
]
assert processed_approval_inputs == [
    'business_stakeholder_processed_review',
    'ml_modeler_processed_review',
    'ml_reviewer_processed_review',
]

print('\nFan-out / fan-in checks passed.')


Raw fan-out targets:
['business_stakeholder_raw_review',
 'ml_modeler_raw_review',
 'ml_reviewer_raw_review']

Processed fan-out targets:
['business_stakeholder_processed_review',
 'ml_modeler_processed_review',
 'ml_reviewer_processed_review']

Fan-in to eda_prep_plan:
['business_stakeholder_raw_review',
 'data_engineer_feedback',
 'eda_processed_approval',
 'ml_modeler_raw_review',
 'ml_reviewer_raw_review']

Fan-in to eda_processed_approval:
['business_stakeholder_processed_review',
 'ml_modeler_processed_review',
 'ml_reviewer_processed_review']

Fan-out / fan-in checks passed.


In [5]:
from typing import Any
from unittest.mock import patch

from multi_agent_ds.agents.business_stakeholder import business_stakeholder_node
from multi_agent_ds.agents.eda_analyst import eda_analyst_node
from multi_agent_ds.agents.ml_modeler import ml_modeler_node
from multi_agent_ds.agents.ml_reviewer import ml_reviewer_node


class FakeAdapter:
    def __init__(self, settings: dict[str, Any]):
        self.settings = settings

    def structured_output(self, messages, schema):
        properties = schema['properties']

        if 'needs_cleaning' in properties:
            return {
                'parsed': {
                    'n_rows': 4,
                    'n_features': 2,
                    'target_rate': 0.5,
                    'needs_cleaning': True,
                    'feature_summaries': [{'feature': 'age', 'issue': 'right_skew'}],
                    'correlation_flags': [],
                    'recommendations': ['Inspect skewed numeric features before modeling.'],
                }
            }

        return {
            'parsed': {
                'reviewer_role': 'ml_reviewer',
                'summary': 'EDA review looks acceptable.',
                'concerns': [],
                'recommendations': ['Proceed carefully.'],
                'modeling_implications': ['Use robust validation.'],
                'business_implications': [],
                'scientific_vs_art': [],
            }
        }


def fake_prompts():
    return {
        'eda_analyst': {
            'system': 'system prompt',
            'raw_review': 'rows={n_rows}; features={n_features}; target_rate={target_rate}; {profile_json}',
            'prep_plan': 'raw={raw_eda_json}; modeler={ml_modeler_review_json}; review={ml_review_json}; business={business_review_json}; feedback={prep_feedback_json}',
            'processed_approval': 'processed={processed_eda_json}; modeler={ml_modeler_review_json}; review={ml_review_json}; business={business_review_json}',
        },
        'sean_ml_modeler': {
            'system': 'You are an ML modeler agent.',
            'eda_review': 'stage={review_stage}; eda={eda_json}',
        },
        'ml_reviewer': {
            'system': 'You are an ML reviewer.',
            'eda_review': 'stage={review_stage}; eda={eda_json}',
        },
        'business_stakeholder': {
            'system': 'You are a business stakeholder.',
            'eda_review': 'stage={review_stage}; eda={eda_json}',
        },
    }


profile_result = {
    'data_path': 'data/raw/sample.parquet',
    'target_column': 'binary_target',
    'n_rows': 4,
    'n_features': 2,
    'profile': {
        'dataset_summary': {
            'n_rows': 4,
            'n_features': 2,
            'target_column': 'binary_target',
            'excluded_metadata_columns': [],
            'numeric_features': ['age'],
            'categorical_features': ['state'],
        },
        'distributions': {'feature_counts': {'numeric': 1, 'categorical': 1}},
        'target_analysis': {'positive_rate': 0.5},
        'correlations': {'high_correlation_pairs': [], 'target_correlations': []},
        'feature_target_relationships': {
            'top_numerical_signals': [],
            'top_categorical_signals': [],
        },
        'outliers': {'flagged_features': []},
    },
}

settings = {
    'llm': {'providers': {'openai': {'model': 'gpt-4o', 'temperature': 0.2, 'max_tokens': 1000}}}
}

with (
    patch('multi_agent_ds.agents.eda_analyst.OpenAIAdapter', FakeAdapter),
    patch('multi_agent_ds.agents.ml_modeler.OpenAIAdapter', FakeAdapter),
    patch('multi_agent_ds.agents.ml_reviewer.OpenAIAdapter', FakeAdapter),
    patch('multi_agent_ds.agents.business_stakeholder.OpenAIAdapter', FakeAdapter),
    patch('multi_agent_ds.agents.eda_analyst.load_prompts_config', fake_prompts),
    patch('multi_agent_ds.agents.ml_modeler.load_prompts_config', fake_prompts),
    patch('multi_agent_ds.agents.ml_reviewer.load_prompts_config', fake_prompts),
    patch('multi_agent_ds.agents.business_stakeholder.load_prompts_config', fake_prompts),
):
    raw_state = {
        'settings': settings,
        'data': profile_result,
        'agent_decisions': [],
    }
    raw_result = eda_analyst_node(raw_state, mode='raw')
    modeler_review = ml_modeler_node({'settings': settings, 'raw_eda_insights': raw_result['raw_eda_insights'], 'agent_decisions': []}, mode='raw_review')
    ml_review = ml_reviewer_node({'settings': settings, 'raw_eda_insights': raw_result['raw_eda_insights'], 'agent_decisions': []}, mode='raw_review')
    business_review = business_stakeholder_node({'settings': settings, 'raw_eda_insights': raw_result['raw_eda_insights'], 'agent_decisions': []}, mode='raw_review')

print('Raw EDA output:')
pprint(raw_result['raw_eda_insights'])
print('\nML modeler review:')
pprint(modeler_review['raw_eda_ml_modeler_review'])
print('\nML reviewer review:')
pprint(ml_review['raw_eda_ml_review'])
print('\nBusiness stakeholder review:')
pprint(business_review['raw_eda_business_review'])


Raw EDA output:
{'correlation_flags': [],
 'feature_summaries': [{'feature': 'age', 'issue': 'right_skew'}],
 'n_features': 2,
 'n_rows': 4,
 'needs_cleaning': True,
 'recommendations': ['Inspect skewed numeric features before modeling.'],
 'target_rate': 0.5}

ML modeler review:
{'business_implications': [],
 'concerns': [],
 'modeling_implications': ['Use robust validation.'],
 'recommendations': ['Proceed carefully.'],
 'reviewer_role': 'ml_reviewer',
 'scientific_vs_art': [],
 'summary': 'EDA review looks acceptable.'}

ML reviewer review:
{'business_implications': [],
 'concerns': [],
 'modeling_implications': ['Use robust validation.'],
 'recommendations': ['Proceed carefully.'],
 'reviewer_role': 'ml_reviewer',
 'scientific_vs_art': [],
 'summary': 'EDA review looks acceptable.'}

Business stakeholder review:
{'business_implications': [],
 'concerns': [],
 'modeling_implications': ['Use robust validation.'],
 'recommendations': ['Proceed carefully.'],
 'reviewer_role': 'ml_revie

## LangSmith

If you want a UI for runtime traces, enable LangSmith in your environment:

```bash
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=your-key
export LANGSMITH_PROJECT=multi_agent_ds_graph_debug
```

Then run the graph normally and inspect the trace UI for per-node execution, timing, inputs, and outputs.
